In [1]:
%%capture
%pip install pytube 
%pip install youtube-transcript-api==1.1.0
%pip install langchain-community==0.3.16
%pip install langchain==0.3.23
%pip install langchain-openai==0.3.14
%pip install yt-dlp

In [1]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display, JSON
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
import json

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress pytube errors
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suppress yt-dlp warnings
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [3]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [4]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11-character YouTube video ID from a URL.

Args:
    url (str): A YouTube URL containing a video ID.

Returns:
    str: Extracted video ID or error message if parsing fails.
----------------------------
<function extract_video_id at 0x71c39bff11c0>

In [5]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [6]:
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x71c39bff11c0>)

In [7]:
tools = []
tools.append(extract_video_id)

In [8]:
from youtube_transcript_api import YouTubeTranscriptApi


@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [9]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [10]:
tools.append(fetch_transcript)

In [11]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
search_out=search_youtube.run("Generative AI")
display(JSON(search_out))

<IPython.core.display.JSON object>

In [13]:
tools.append(search_youtube)

In [14]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [15]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

In [16]:
tools.append(get_full_metadata)

In [17]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [18]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails))

<IPython.core.display.JSON object>

In [19]:
tools.append(get_thumbnails)

In [20]:
llm_with_tools = llm.bind_tools(tools)

In [21]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))
    

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [22]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english

In [23]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]

In [24]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_BLltWanwuyDi7rioPlZohMuA', 'function': {'arguments': '{"url":"https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'extract_video_id'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 380, 'total_tokens': 409, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0b9898f509', 'id': 'chatcmpl-ETr3U8zV3rw4u7di3BlBvhS65rNHk', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a0f30d-cb78-72b0-81ad-7781408cca1c-0', tool_calls=[{'name': 'extract_video_id', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'call_BLltWanwuyDi7rioPlZohMuA', 'type': 'tool_call'}], usage_metadata={'input_tokens': 380, 'output_toke

In [25]:
messages.append(response_1)

In [26]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [27]:
tool_calls_1 = response_1.tool_calls
display(JSON(tool_calls_1))

<IPython.core.display.JSON object>

In [28]:
tool_name=tool_calls_1[0]['name']
print(tool_name)

extract_video_id

In [29]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

call_BLltWanwuyDi7rioPlZohMuA

In [30]:
args=tool_calls_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}

In [31]:
my_tool=tool_mapping[tool_calls_1[0]['name']]

In [32]:
video_id =my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [33]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

In [34]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_h1NYuFJ9PgHHw5mCe2mF4jK7', 'function': {'arguments': '{"video_id":"T-D1OfcDW1M","language":"en"}', 'name': 'fetch_transcript'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 426, 'total_tokens': 453, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0b9898f509', 'id': 'chatcmpl-ETr65DpRCmRZel0pTEpvFQwyoumce', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a0f310-43f7-7632-9fa4-709e101da761-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'}, 'id': 'call_h1NYuFJ9PgHHw5mCe2mF4jK7', 'type': 'tool_call'}], usage_metadata={'input_tokens': 426, 'output_tokens': 27, 'total_toke

In [35]:
messages.append(response_2)

In [36]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'call_h1NYuFJ9PgHHw5mCe2mF4jK7',
  'type': 'tool_call'}]

In [37]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

In [38]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [39]:
summary = llm_with_tools.invoke(messages)

In [40]:
summary

AIMessage(content='The video features Marina Danilevsky from IBM Research discussing a framework called Retrieval-Augmented Generation (RAG) designed to enhance the accuracy and relevance of large language models (LLMs).\n\n### Summary:\n\n1. **Introduction to LLMs**: The speaker highlights the omnipresence of large language models and their ability to provide both accurate and inaccurate information.\n\n2. **Problems with LLMs**: \n   - Lack of sourcing: LLMs can provide confident answers without citing reliable sources.\n   - Outdated information: They may offer information that is no longer accurate due to new discoveries.\n\n3. **Example**: Danilevsky shares a personal anecdote about answering a question regarding the planet with the most moons, initially stating Jupiter based on outdated knowledge without sourcing.\n\n4. **RAG Framework Explanation**:\n   - Instead of solely relying on pre-trained data, RAG allows the LLM to retrieve information from a content store (like the inte

In [41]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

        

In [42]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [43]:
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [44]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 The video provides an overview of LangChain, an open-source orchestration framework designed for developing applications that utilize large language models (LLMs). Launched in October 2022, LangChain allows developers to create applications that can seamlessly switch between different LLMs and integrate them with data sources and workflows.

Key components of LangChain include:

1. **LLM Module**: A standard interface for various LLMs, allowing developers to choose between closed-source or open-source models.
2. **Prompt Templates**: Tools that simplify the creation of prompts for the LLM without hardcoding queries.
3. **Chains**: Sequences that combine different functions to execute complex NLP tasks, where the output of one function feeds into the next.
4. **Indexes and Document Loaders**: Mechanisms to retrieve specific external data sources, like documents or web content.
5. **Memory Utilities**: Options to retain long-term memory of prior conversations, enhancing a

In [45]:
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

In [46]:
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

In [47]:
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
)

In [48]:
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

In [49]:
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
)

In [50]:
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"])

In [51]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [52]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = summarization_chain.invoke(query)
print("Video Summary:\n", result)

Video Summary:
 The video discusses a framework called Retrieval-Augmented Generation (RAG) aimed at enhancing the accuracy and currency of large language models (LLMs). Marina Danilevsky, a Senior Research Scientist at IBM Research, explains how LLMs can sometimes provide incorrect or outdated information confidently.

She shares a personal anecdote where she mistakenly answered a question about which planet in the solar system has the most moons, highlighting two main issues typical of LLMs: the lack of sourcing and the risk of providing outdated information. Using her example, she illustrates that a standard LLM might confidently assert that Jupiter has the most moons without verifying this against current, reliable sources.

In contrast, RAG improves this by integrating a retrieval mechanism. When a user asks a question, the LLM first retrieves relevant information from a content store (which can be open, like the internet, or closed, like certain document collections) before gener

In [53]:
query = {"query": "Get top 3 youtube videos in India and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Video Summary:
 Here are the top 3 YouTube videos in India along with their metadata:

### 1. [Wonders of India | Most Amazing Places in India | 4K Travel Guide](https://youtu.be/xUphsn5b1w8)
- **Channel**: EpicExplorationsTV EN
- **Views**: 1,886,503
- **Duration**: 1 hour, 6 minutes, 39 seconds
- **Likes**: 8,412
- **Comments**: 399
- **Chapters**:
  - Introduction (0:00 - 0:42)
  - Above India (0:42 - 7:23)
  - Taj Mahal (7:23 - 8:57)
  - Varanasi (8:57 - 10:33)
  - Kerala (10:33 - 12:18)
  - ... (and many more)

---

### 2. [Wonders of India | The Most Amazing Places in India | 4K Travel Documentary](https://youtu.be/FM2OuVbkPVw)
- **Channel**: Explore Nature Today
- **Views**: 1,250,401
- **Duration**: 46 minutes, 28 seconds
- **Likes**: 12,197
- **Comments**: 448
- **Chapters**:
  - Introduction to India (0:00 - 1:16)
  - Statue of unity (1:16 - 2:28)
  - Auroville (2:28 - 3:03)
  - Stakna gompa (3:03 - 3:58)
  - Tea and wildlife in Kerala (3:58 - 5:51)
  - ... (and many more)

-

In [54]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

In [55]:
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]

In [56]:
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))

In [57]:
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

In [58]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [60]:
query_us = {"query": "Show top 3 India trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)

KeyboardInterrupt: 